In [0]:
%pip install google-api-python-client google-auth


In [0]:
import json
with open("/Volumes/dev_p2p/staging/raw_data/GoogleDriveCredentials/testproject1-501911-eea1aa67b41e.json") as f:
       creds_dict = json.load(f)

In [0]:
from google.oauth2 import service_account
from googleapiclient.discovery import build

# Build credentials from the loaded dict
creds = service_account.Credentials.from_service_account_info(
    creds_dict,
    scopes=["https://www.googleapis.com/auth/drive.readonly"]
)

# Connect to the Drive API
drive_service = build("drive", "v3", credentials=creds)

# Test the connection — list files visible to the service account
results = drive_service.files().list(
    pageSize=20,
    fields="files(id, name, mimeType)"
).execute()

files = results.get("files", [])
if not files:
    print("No files found. Did you share a folder with the service account's email?")
else:
    for f in files:
        print(f"{f['name']}  |  {f['id']}  |  {f['mimeType']}")

In [0]:
import os
import io
import json
from googleapiclient.http import MediaIoBaseDownload

GOOGLE_EXPORT_MAP = {
    "application/vnd.google-apps.document": ("application/pdf", ".pdf"),
    "application/vnd.google-apps.spreadsheet": ("text/csv", ".csv"),
    "application/vnd.google-apps.presentation": ("application/pdf", ".pdf"),
}
FOLDER_MIME = "application/vnd.google-apps.folder"

MANIFEST_PATH = "/Volumes/dev_p2p/staging/raw_data/_sync_manifest.json"

def load_manifest():
    """Load record of {file_id: modifiedTime} for previously synced files."""
    if os.path.exists(MANIFEST_PATH):
        with open(MANIFEST_PATH, "r") as f:
            return json.load(f)
    return {}

def save_manifest(manifest):
    os.makedirs(os.path.dirname(MANIFEST_PATH), exist_ok=True)
    with open(MANIFEST_PATH, "w") as f:
        json.dump(manifest, f, indent=2)

def list_children(folder_id):
    """List all files/folders directly inside a given Drive folder, including modifiedTime."""
    items = []
    page_token = None
    while True:
        response = drive_service.files().list(
            q=f"'{folder_id}' in parents and trashed = false",
            fields="nextPageToken, files(id, name, mimeType, modifiedTime)",
            pageToken=page_token
        ).execute()
        items.extend(response.get("files", []))
        page_token = response.get("nextPageToken")
        if not page_token:
            break
    return items

def download_file(file_id, mime_type, dest_path):
    if mime_type in GOOGLE_EXPORT_MAP:
        export_mime, ext = GOOGLE_EXPORT_MAP[mime_type]
        if not dest_path.endswith(ext):
            dest_path += ext
        request = drive_service.files().export_media(fileId=file_id, mimeType=export_mime)
    else:
        request = drive_service.files().get_media(fileId=file_id)

    buffer = io.BytesIO()
    downloader = MediaIoBaseDownload(buffer, request)
    done = False
    while not done:
        _, done = downloader.next_chunk()

    with open(dest_path, "wb") as f:
        f.write(buffer.getvalue())
    print(f"Downloaded: {dest_path}")

def sync_folder(folder_id, local_path, manifest, stats):
    os.makedirs(local_path, exist_ok=True)
    items = list_children(folder_id)

    for item in items:
        item_path = os.path.join(local_path, item["name"])

        if item["mimeType"] == FOLDER_MIME:
            sync_folder(item["id"], item_path, manifest, stats)
        else:
            file_id = item["id"]
            modified_time = item["modifiedTime"]
            previous_modified = manifest.get(file_id)

            if previous_modified == modified_time:
                stats["skipped"] += 1
                continue  # unchanged since last sync

            try:
                download_file(file_id, item["mimeType"], item_path)
                manifest[file_id] = modified_time  # update manifest on success
                stats["downloaded"] += 1
            except Exception as e:
                print(f"Failed to download {item['name']}: {e}")
                stats["failed"] += 1

# --- Run it ---
root_folder_id = "1MpPeFV7PXFuFMdj1sJbfg9FRwFkBIqW5"  # from the Drive URL
volume_base_path = "/Volumes/dev_p2p/staging/raw_data/"

manifest = load_manifest()
stats = {"downloaded": 0, "skipped": 0, "failed": 0}

sync_folder(root_folder_id, volume_base_path, manifest, stats)
save_manifest(manifest)

print(f"\nSync complete: {stats['downloaded']} downloaded, {stats['skipped']} unchanged, {stats['failed']} failed")